# Más transformaciones con la API DataFrame

Continuamos con más ejemplos útiles: seleccionar campos, añadir columnas calculadas, ordernar, renombrar y eliminar columnas, valores únicos, agregaciones, condiciones lógicas, columnas condicionales (`when`), casting de tipos y funciones de fecha.


In [0]:
# Leemos la tabla de Unity Catalog y la guardamos en un DataFrame
# spark.read.table carga la tabla completa en memoria (de forma perezosa, lazy)
df_clientes_compras = spark.read.table("workspace.default.fct_clientes_compras")

# display() muestra el contenido del DataFrame en una tabla interactiva
df_clientes_compras.display()

In [0]:
# Otra forma de leer los mismos datos: escribiendo SQL directamente
# spark.sql() devuelve el resultado de la consulta como un DataFrame
df_clientes_compras = spark.sql("select * from workspace.default.fct_clientes_compras")

df_clientes_compras.display()

In [0]:
# count() devuelve el numero total de filas del DataFrame
# Con display() vemos los datos; con count() obtenemos un unico numero
df_clientes_compras.count()

In [0]:
# select(): nos quedamos solo con las columnas que nos interesan
# Los nombres de columna se pasan como texto, entre comillas
df_clientes_compras.select("productId", "producto", "nombre").display()

In [0]:
# Misma consulta que antes, pero usando la funcion col()
# col("productId") es lo mismo que escribir "productId"
# col() es util cuando queremos aplicar funciones o condiciones sobre la columna
from pyspark.sql.functions import col

df_clientes_compras.select(col("productId"), col("producto"), col("nombre")).display()


In [0]:
# filter(): se queda solo con las filas que cumplen la condicion
# Aqui: las compras del cliente llamado "Nuria"
df_clientes_compras.filter(col("nombre")=="Nuria").display()

In [0]:
# Misma consulta que antes, pero sin usar col()
# Tambien podemos acceder a la columna con df_clientes_compras.nombre (notacion punto)
df_clientes_compras.filter(df_clientes_compras.nombre=="Nuria").display()

In [0]:
# withColumn() añade una columna NUEVA al DataFrame
# lit() se usa para poner un valor fijo (igual) en todas las filas
from pyspark.sql.functions import lit

df_clientes_compras.withColumn("nueva_columna", lit("hola!")).display()

In [0]:
# withColumn tambien puede crear una columna combinando otras
# concat() une varias columnas y textos en una sola columna
from pyspark.sql.functions import concat

df_clientes_compras.withColumn("pais y ciudad", concat(col("pais"), lit(" - "), col("ciudad"))).display()

In [0]:
# withColumn con una funcion de fecha: current_timestamp()
# La columna nueva se rellena con la fecha y hora actual en cada fila
from pyspark.sql.functions import current_timestamp

df_clientes_compras.withColumn("fecha_procesamiento", current_timestamp()).display()

In [0]:
# sort() ordena las filas por una columna; desc() = de mayor a menor
# limit(5) se queda solo con las 5 primeras filas (los 5 productos mas caros)
df_clientes_compras.sort(col("precioUnitario").desc()).limit(5).display()

In [0]:
# Igual que antes pero con asc(): de menor a mayor (los 5 productos mas baratos)
df_clientes_compras.sort(col("precioUnitario").asc()).limit(5).display()

In [0]:
# groupBy() agrupa las filas por el valor de una columna
# count() cuenta cuantas filas hay en cada grupo (aqui: numero de compras por producto)
df_clientes_compras.groupBy("producto").count().display()

In [0]:
from pyspark.sql.functions import col

# Renombrar una columna: withColumnRenamed("nombre_actual", "nombre_nuevo")
df_clientes_compras.withColumnRenamed("producto", "nombre_producto").select("nombre_producto", "cantidad").display()

In [0]:
# Eliminar columnas que no necesitamos con drop()
df_clientes_compras.drop("latitud", "longitud").display()

In [0]:
# Valores únicos: distinct() devuelve las filas sin duplicados
df_clientes_compras.select("pais", "comunidadAutonoma").distinct().display()

# También podemos quitar duplicados solo mirando una columna concreta
df_clientes_compras.select("categoria").dropDuplicates(["categoria"]).display()

In [0]:
from pyspark.sql.functions import sum, avg, min, max, count, round

# Agregaciones sobre todo el DataFrame: total vendido, ticket medio, precio maximo y minimo
df_clientes_compras.agg(
    count("purchaseId").alias("total_compras"),
    sum("importeTotal").alias("importe_total_vendido"),
    round(avg("precioUnitario"), 2).alias("precio_medio"),
    min("fechaCompra").alias("primera_compra"),
    max("fechaCompra").alias("ultima_compra")
).display()

In [0]:
# groupBy + agg: agregar por grupo. Aqui vemos cuanto se vende por categoria
df_clientes_compras.groupBy("categoria").agg(
    count("purchaseId").alias("num_compras"),
    sum("cantidad").alias("unidades_vendidas"),
    round(avg("importeTotal"), 2).alias("importe_medio")
).display()

In [0]:
# Filtros con varias condiciones: & (AND) y | (OR). ¡Cada condición necesita sus paréntesis!

# Compras de la categoria "Electronica" con importe mayor de 500
df_clientes_compras.filter((col("categoria") == "Electronica") & (col("importeTotal") > 500)).display()

# Compras de España o Portugal
df_clientes_compras.filter((col("pais") == "España") | (col("pais") == "Portugal")).display()

In [0]:
# isin: filtrar por una lista de valores, mucho mas comodo que encadenar OR
df_clientes_compras.filter(col("nombre").isin("Nuria", "Juan", "Ana")).display()

# isNotNull: quedarnos solo con las filas que SI tienen ciudad informada
df_clientes_compras.filter(col("ciudad").isNotNull()).count()

In [0]:
from pyspark.sql.functions import when

# when / otherwise: crear una columna con valores condicionales (como un IF-ELSE)
df_clientes_compras.withColumn(
    "tamano_compra",
    when(col("importeTotal") >= 1000, "GRANDE")
    .when(col("importeTotal") >= 250, "MEDIANA")
    .otherwise("PEQUEÑA")
).select("nombre", "producto", "importeTotal", "tamano_compra").display()

In [0]:
from pyspark.sql.functions import round as redondear

# Columna calculada: los importes se recalculan con una expresion aritmetica
df_clientes_compras.withColumn(
    "importe_calculado", redondear(col("cantidad") * col("precioUnitario"), 2)
).select("cantidad", "precioUnitario", "importeTotal", "importe_calculado").display()

In [0]:
# cast: cambiar el tipo de una columna (aqui pasamos el importe a ENTERO)
df_clientes_compras.withColumn("importe_entero", col("importeTotal").cast("int")).select("importeTotal", "importe_entero").display()

In [0]:
from pyspark.sql.functions import year, month

# Funciones de fecha: extraer el año y el mes de fechaCompra
df_clientes_compras.withColumn("anio", year(col("fechaCompra"))).withColumn("mes", month(col("fechaCompra"))).select("fechaCompra", "anio", "mes").display()

In [0]:
from pyspark.sql.functions import col, sum

# orderBy con varias columnas: primero por unidades vendidas (desc) y luego por categoria (asc)
df_clientes_compras.groupBy("categoria").agg(sum("cantidad").alias("unidades_vendidas")) \
    .orderBy(col("unidades_vendidas").desc(), col("categoria").asc()).display()

In [0]:
from pyspark.sql.functions import upper, lower, initcap, length, substring

# Funciones de texto sobre la columna ciudad
df_clientes_compras.select("ciudad") \
    .withColumn("mayusculas", upper(col("ciudad"))) \
    .withColumn("minusculas", lower(col("ciudad"))) \
    .withColumn("capitalizada", initcap(col("ciudad"))) \
    .withColumn("longitud", length(col("ciudad"))) \
    .withColumn("tres_primeras_letras", substring(col("ciudad"), 1, 3)) \
    .limit(5).display()

# Resumen de lo aprendido

| Transformación | Método | Ejemplo |
| --- | --- | --- |
| Seleccionar columnas | `select()` | `df.select("producto")` |
| Filtrar filas | `filter()` / `where()` | `df.filter(col("pais") == "España")` |
| Añadir / cambiar columna | `withColumn()` | `df.withColumn("nueva", ...)` |
| Renombrar columna | `withColumnRenamed()` | `df.withColumnRenamed("a", "b")` |
| Eliminar columna | `drop()` | `df.drop("latitud")` |
| Valores únicos | `distinct()` | `df.select("pais").distinct()` |
| Agrupar y agregar | `groupBy().agg()` | `df.groupBy("categoria").agg(sum(...))` |
| Ordenar | `sort()` / `orderBy()` | `df.sort(col("x").desc())` |
| Limitar filas | `limit()` | `df.limit(5)` |
| Columna condicional | `when() / otherwise()` | `when(col("x") > 1, "S").otherwise("N")` |
| Cambiar tipo | `cast()` | `col("x").cast("int")` |